# Class Project U3

**Scarlette Berenice Alcaraz Muñiz**  

Maestría en Ciencias Computacionales  

**Materia:** Aprendizaje por Refuerzo  

**Docente:** Dr. Julio Alberto García Rodríguez

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

np.random.seed(42)

# ── MDP Parameters
ALPHA    = 0.7
BETA     = 0.4
R_SEARCH = 3.0
R_WAIT   = 1.0
R_RESCUE = -3.0

HIGH = 'high'
LOW  = 'low'

STEPS_PER_EPISODE = 100
NUM_EPISODES      = 50

# ── Color Palette
C_HIGH     = '#2196F3'
C_LOW      = '#FF9800'
C_REWARD   = '#4CAF50'
C_RESCUE   = '#F44336'
C_GREEDY   = '#E53935'
C_BALANCED = '#1E88E5'

print('✅ Configuration loaded.')
print(f'   α={ALPHA}, β={BETA}, r_search={R_SEARCH}, r_wait={R_WAIT}')


## Task 1 

In [ ]:
def step(state, action, alpha=ALPHA, beta=BETA):
    """Executes an MDP transition. Returns (next_state, reward, rescued)."""
    rescued = False

    if state == HIGH and action == 'search':
        next_state = HIGH if np.random.random() < alpha else LOW
        reward     = R_SEARCH

    elif state == HIGH and action == 'wait':
        next_state = HIGH
        reward     = R_WAIT

    elif state == LOW and action == 'search':
        if np.random.random() < beta:
            next_state = LOW
            reward     = R_SEARCH
        else:
            next_state = HIGH
            reward     = R_RESCUE
            rescued    = True

    elif state == LOW and action == 'wait':
        next_state = LOW
        reward     = R_WAIT

    elif state == LOW and action == 'recharge':
        next_state = HIGH
        reward     = 0.0

    else:
        raise ValueError(f"Invalid combination: state='{state}', action='{action}'")

    return next_state, reward, rescued


def simulate_episode(policy_fn, steps=STEPS_PER_EPISODE,
                     alpha=ALPHA, beta=BETA, init_state=HIGH):
    """Simulates a complete episode following a given policy."""
    state   = init_state
    history = {
        'states':            [state],
        'actions':           [],
        'rewards':           [],
        'rescued':           [],
        'cumulative_reward': [0.0]
    }
    for _ in range(steps):
        action                      = policy_fn(state)
        next_state, reward, rescued = step(state, action, alpha, beta)
        history['states'].append(next_state)
        history['actions'].append(action)
        history['rewards'].append(reward)
        history['rescued'].append(rescued)
        history['cumulative_reward'].append(
            history['cumulative_reward'][-1] + reward
        )
        state = next_state
    return history


# ── Quick test ───────────────────────────────────────────────────────────────
print('✅ Simulation functions defined.')
print('\n── Test: 10 steps from HIGH with search action ──')
state = HIGH
for i in range(10):
    ns, r, rescued = step(state, 'search')
    print(f'  Step {i+1}: {state} → {ns} | reward={r:+.1f} | rescued={rescued}')
    state = ns

In [ ]:
# ── Greedy Policy (used in Tasks 1 and 2) ────────────────────────────────────
def greedy_policy(state):
    return 'search'

# ── Impact of α and β ────────────────────────────────────────────────────────
configs = [
    (0.9, 0.7, 'High α, High β'),
    (0.7, 0.4, 'Medium α, Medium β (baseline)'),
    (0.3, 0.2, 'Low α, Low β'),
]

print(f"{'Configuration':<35} {'Avg. Reward':>15} {'Avg. Rescues':>16}")
print('-' * 70)
for alpha_v, beta_v, label in configs:
    rewards  = [simulate_episode(greedy_policy, alpha=alpha_v, beta=beta_v)['cumulative_reward'][-1]
                for _ in range(NUM_EPISODES)]
    rescates = [sum(simulate_episode(greedy_policy, alpha=alpha_v, beta=beta_v)['rescued'])
                for _ in range(NUM_EPISODES)]
    print(f'  {label:<33} {np.mean(rewards):>14.2f} {np.mean(rescates):>15.2f}')

## Task 2 

In [ ]:
# ── Graph 1: State frequency (reference episode)
ref_history  = simulate_episode(greedy_policy)
states_seq   = ref_history['states'][1:]
freq_high    = states_seq.count(HIGH)
freq_low     = states_seq.count(LOW)

fig_freq = go.Figure()
fig_freq.add_trace(go.Bar(
    x=[HIGH, LOW],
    y=[freq_high, freq_low],
    marker_color=[C_HIGH, C_LOW],
    marker_line_color='white',
    marker_line_width=2,
    text=[f'{freq_high} ({freq_high}%)', f'{freq_low} ({freq_low}%)'],
    textposition='outside',
    hovertemplate='State: <b>%{x}</b><br>Steps: <b>%{y}</b><extra></extra>',
    width=0.4
))
fig_freq.update_layout(
    title=dict(text=f'Task 2 — State Frequency<br><sup>α={ALPHA}, β={BETA}, Greedy policy, {STEPS_PER_EPISODE} steps</sup>',
               font_size=15),
    xaxis_title='Battery State',
    yaxis_title='Number of steps',
    yaxis_range=[0, STEPS_PER_EPISODE * 1.25],
    plot_bgcolor='white',
    height=420,
    showlegend=False
)
fig_freq.update_xaxes(showgrid=False)
fig_freq.update_yaxes(showgrid=True, gridcolor='#eee')
fig_freq.show()

In [ ]:
# ── Graph 2: Cumulative reward + rescues ──────────────────────────────────────
cum_r        = ref_history['cumulative_reward']
rescue_steps = [i+1 for i, r in enumerate(ref_history['rescued']) if r]
rescue_vals  = [cum_r[s] for s in rescue_steps]
actions_seq  = ref_history['actions']

hover_text = []
for i, (s, a, rw) in enumerate(zip(
        ref_history['states'][1:],
        actions_seq,
        ref_history['rewards'])):
    hover_text.append(
        f'Step: {i+1}<br>State: {s}<br>Action: {a}<br>Reward: {rw:+.1f}<br>Cum.: {cum_r[i+1]:.1f}'
    )

fig_rew = go.Figure()
fig_rew.add_trace(go.Scatter(
    x=list(range(len(cum_r))),
    y=cum_r,
    mode='lines',
    name='Cumulative reward',
    line=dict(color=C_REWARD, width=2.5),
    hovertemplate='Step %{x}<br>Cum.: <b>%{y:.1f}</b><extra></extra>'
))

if rescue_steps:
    fig_rew.add_trace(go.Scatter(
        x=rescue_steps,
        y=rescue_vals,
        mode='markers',
        name=f'Rescue ({len(rescue_steps)}x)',
        marker=dict(color=C_RESCUE, size=10, symbol='triangle-down',
                    line=dict(color='white', width=1)),
        hovertemplate='Step %{x}<br>Penalty −3<br>Cum.: %{y:.1f}<extra></extra>'
    ))

fig_rew.update_layout(
    title=dict(text='Task 2 — Cumulative Reward Over Time<br><sup>Hover over the line to see details for each step</sup>',
               font_size=15),
    xaxis_title='Time step',
    yaxis_title='Cumulative reward',
    plot_bgcolor='white',
    height=450,
    hovermode='x unified',
    legend=dict(x=0.01, y=0.99)
)
fig_rew.update_xaxes(showgrid=True, gridcolor='#eee')
fig_rew.update_yaxes(showgrid=True, gridcolor='#eee')
fig_rew.show()

In [ ]:
# ── Graph 3: Configuration comparison (α, β) ─────────────────────────────────
configs_plot = [
    (0.9, 0.7, '#1565C0', 'α=0.9, β=0.7 (optimistic)'),
    (0.7, 0.4, '#43A047', 'α=0.7, β=0.4 (baseline)'),
    (0.5, 0.5, '#FB8C00', 'α=0.5, β=0.5 (symmetric)'),
    (0.3, 0.2, '#E53935', 'α=0.3, β=0.2 (pessimistic)'),
]

fig_ab = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Average Cumulative Reward (±1σ)', 'Average Rescues per Episode')
)

mean_rescues = []
labels_short = []

for alpha_v, beta_v, color, label in configs_plot:
    all_cum  = []
    all_resc = []
    for _ in range(NUM_EPISODES):
        h = simulate_episode(greedy_policy, alpha=alpha_v, beta=beta_v)
        all_cum.append(h['cumulative_reward'])
        all_resc.append(sum(h['rescued']))

    mean_c = np.mean(all_cum, axis=0)
    std_c  = np.std(all_cum,  axis=0)
    x      = list(range(len(mean_c)))

    # Confidence band
    fig_ab.add_trace(go.Scatter(
        x=x + x[::-1],
        y=list(mean_c + std_c) + list((mean_c - std_c)[::-1]),
        fill='toself', fillcolor=color, opacity=0.12,
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False, hoverinfo='skip'
    ), row=1, col=1)

    # Average line
    fig_ab.add_trace(go.Scatter(
        x=x, y=mean_c, mode='lines',
        name=label, line=dict(color=color, width=2),
        hovertemplate=f'{label}<br>Step %{{x}}<br>Cum.: %{{y:.1f}}<extra></extra>'
    ), row=1, col=1)

    mean_rescues.append(np.mean(all_resc))
    labels_short.append(f'α={alpha_v}, β={beta_v}')

# Rescue bars
fig_ab.add_trace(go.Bar(
    x=labels_short,
    y=mean_rescues,
    marker_color=[c for _, _, c, _ in configs_plot],
    marker_line_color='white',
    marker_line_width=2,
    text=[f'{v:.1f}' for v in mean_rescues],
    textposition='outside',
    showlegend=False,
    hovertemplate='%{x}<br>Avg. rescues: <b>%{y:.1f}</b><extra></extra>'
), row=1, col=2)

fig_ab.update_layout(
    title=dict(text=f'Task 2 — Impact of α and β — Greedy Policy ({NUM_EPISODES} episodes × {STEPS_PER_EPISODE} steps)',
               font_size=14),
    plot_bgcolor='white',
    height=480,
    hovermode='x'
)
for axis in ['xaxis', 'xaxis2', 'yaxis', 'yaxis2']:
    fig_ab.update_layout(**{axis: dict(showgrid=True, gridcolor='#eee')})
fig_ab.show()

## Task 3

| Policy | In `high` | In `low` | Characteristic |
|---------|------------|-----------|----------------|
| **Greedy** | search | search | Maximizes collection, high rescue risk |
| **Balanced** | search | recharge | Takes advantage of `high`; in `low` it recharges before taking risks |

In [ ]:
# ── Policy definitions ────────────────────────────────────────────────────────

def policy_greedy(state):
    """Always searches regardless of battery level."""
    return 'search'

def policy_balanced(state):
    """Searches from HIGH; recharges from LOW."""
    return 'search' if state == HIGH else 'recharge'

POLICIES = {'Greedy': policy_greedy, 'Balanced': policy_balanced}
P_COLORS  = {'Greedy': C_GREEDY,     'Balanced': C_BALANCED}

print('✅ Policies defined.')

In [ ]:
# ── Evaluation: 50 episodes per policy ───────────────────────────────────────
results = {}
for name, fn in POLICIES.items():
    ep_rewards  = []
    ep_rescues  = []
    ep_high_pct = []
    all_curves  = []
    for _ in range(NUM_EPISODES):
        h = simulate_episode(fn)
        ep_rewards.append(h['cumulative_reward'][-1])
        ep_rescues.append(sum(h['rescued']))
        ep_high_pct.append(h['states'][1:].count(HIGH) / STEPS_PER_EPISODE * 100)
        all_curves.append(h['cumulative_reward'])
    results[name] = dict(rewards=ep_rewards, rescues=ep_rescues,
                         high_pct=ep_high_pct, cum_curves=all_curves)

print(f"\n{'='*65}")
print(f"  Results ({NUM_EPISODES} episodes × {STEPS_PER_EPISODE} steps)  |  α={ALPHA}, β={BETA}")
print(f"{'='*65}")
print(f"  {'Policy':<12} {'Avg. Reward':>12} {'Reward Std':>11} {'Avg. Rescues':>14} {'% in HIGH':>10}")
print(f"  {'-'*60}")
for name, r in results.items():
    print(f"  {name:<12} "
          f"{np.mean(r['rewards']):>12.2f} "
          f"{np.std(r['rewards']):>11.2f} "
          f"{np.mean(r['rescues']):>14.2f} "
          f"{np.mean(r['high_pct']):>9.1f}%")
print(f"{'='*65}")

In [ ]:
# ── Graph A: Average cumulative reward ───────────────────────────────────────
fig_a = go.Figure()

for name, r in results.items():
    mean_c = np.mean(r['cum_curves'], axis=0)
    std_c  = np.std(r['cum_curves'],  axis=0)
    x      = list(range(len(mean_c)))
    color  = P_COLORS[name]

    # ±1σ band
    fig_a.add_trace(go.Scatter(
        x=x + x[::-1],
        y=list(mean_c + std_c) + list((mean_c - std_c)[::-1]),
        fill='toself', fillcolor=color, opacity=0.15,
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False, hoverinfo='skip'
    ))
    # Average line
    fig_a.add_trace(go.Scatter(
        x=x, y=mean_c, mode='lines',
        name=name, line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{name}</b><br>Step %{{x}}<br>Avg. cum.: %{{y:.1f}}<extra></extra>'
    ))

fig_a.update_layout(
    title=dict(text='Task 3 — A) Average Cumulative Reward (±1σ)<br><sup>Greedy vs Balanced</sup>',
               font_size=14),
    xaxis_title='Time step',
    yaxis_title='Cumulative reward',
    plot_bgcolor='white', height=450,
    hovermode='x unified',
    legend=dict(x=0.02, y=0.98)
)
fig_a.update_xaxes(showgrid=True, gridcolor='#eee')
fig_a.update_yaxes(showgrid=True, gridcolor='#eee')
fig_a.show()

In [ ]:
# ── Graph B: Final reward distribution (Box + points) ────────────────────────
fig_b = go.Figure()

for name, r in results.items():
    fig_b.add_trace(go.Box(
        y=r['rewards'],
        name=name,
        marker_color=P_COLORS[name],
        boxmean='sd',
        boxpoints='all',
        jitter=0.3,
        pointpos=-1.6,
        marker=dict(size=5, opacity=0.5),
        line=dict(width=2),
        hovertemplate=f'<b>{name}</b><br>Reward: %{{y:.1f}}<extra></extra>'
    ))

fig_b.update_layout(
    title=dict(text='Task 3 — B) Final Reward Distribution per Episode<br><sup>Each point is an episode; the dashed line indicates the mean</sup>',
               font_size=14),
    yaxis_title='Total cumulative reward',
    plot_bgcolor='white', height=480,
    showlegend=False
)
fig_b.update_yaxes(showgrid=True, gridcolor='#eee')
fig_b.show()

In [ ]:
# ── Graph C+D: Rescues and % in HIGH (subplots) ───────────────────────────────
names   = list(POLICIES.keys())
colors  = [P_COLORS[n] for n in names]

mean_r  = [np.mean(results[n]['rescues'])  for n in names]
std_r   = [np.std(results[n]['rescues'])   for n in names]
mean_h  = [np.mean(results[n]['high_pct']) for n in names]
std_h   = [np.std(results[n]['high_pct'])  for n in names]

fig_cd = make_subplots(
    rows=1, cols=2,
    subplot_titles=('C) Average Rescues per Episode (±1σ)',
                    'D) % of Time in HIGH State (±1σ)')
)

fig_cd.add_trace(go.Bar(
    x=names, y=mean_r,
    error_y=dict(type='data', array=std_r, visible=True),
    marker_color=colors, marker_line_color='white', marker_line_width=2,
    text=[f'{v:.2f}' for v in mean_r], textposition='outside',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Avg. rescues: %{y:.2f}<extra></extra>'
), row=1, col=1)

fig_cd.add_trace(go.Bar(
    x=names, y=mean_h,
    error_y=dict(type='data', array=std_h, visible=True),
    marker_color=colors, marker_line_color='white', marker_line_width=2,
    text=[f'{v:.1f}%' for v in mean_h], textposition='outside',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>% in HIGH: %{y:.1f}%<extra></extra>'
), row=1, col=2)

fig_cd.update_layout(
    title=dict(text=f'Task 3 — Comparative Metrics | α={ALPHA}, β={BETA}', font_size=14),
    plot_bgcolor='white', height=460
)
fig_cd.update_yaxes(showgrid=True, gridcolor='#eee')
fig_cd.update_layout(yaxis2_range=[0, 115])
fig_cd.show()